# EP01. Hybrid Search 구현하기

1. **BM25 키워드 검색**과 **벡터 검색**의 원리와 차이를 설명할 수 있다
2. `LangChain EnsembleRetriever`로 **RRF 기반 하이브리드 검색**을 구현할 수 있다
3. `Elasticsearch 8.x hybrid query`를 Python으로 실행할 수 있다 (선택)
4. **Recall@10**으로 검색 품질을 측정하고 세 방식을 비교할 수 있다
5. `Langfuse`로 검색 파이프라인을 추적하고 품질 점수를 기록할 수 있다

---
## 섹션 1. 환경 설정

필요한 패키지를 설치합니다. `uv`를 사용하면 훨씬 빠릅니다.

In [1]:
# uv pip install (권장 — pip 대비 10~100x 빠름)
!uv pip install \
    langchain \
    langchain-community \
    langchain-elasticsearch \
    langchain-anthropic \
    langfuse \
    chromadb \
    rank-bm25 \
    sentence-transformers \
    python-dotenv \
    matplotlib \
    pandas \
    --quiet

# 또는 pip를 사용하는 경우:
# !pip install langchain langchain-community langchain-elasticsearch \
#              langchain-anthropic langfuse chromadb rank-bm25 \
#              sentence-transformers python-dotenv matplotlib pandas -q

---
## 섹션 2. 라이브러리 로드 및 API 키 설정

`.env` 파일에서 API 키를 로드합니다. `.env` 파일이 없다면 아래 형식으로 생성하세요:

```
ANTHROPIC_API_KEY=sk-ant-...
LANGFUSE_PUBLIC_KEY=pk-lf-...
LANGFUSE_SECRET_KEY=sk-lf-...
LANGFUSE_HOST=https://cloud.langfuse.com
```

In [2]:
import os
import warnings
warnings.filterwarnings('ignore')

from dotenv import load_dotenv

# .env 파일 로드 (프로젝트 루트 또는 현재 디렉토리)
load_dotenv()

# API 키 확인
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")
LANGFUSE_PUBLIC_KEY = os.getenv("LANGFUSE_PUBLIC_KEY")
LANGFUSE_SECRET_KEY = os.getenv("LANGFUSE_SECRET_KEY")
LANGFUSE_HOST = os.getenv("LANGFUSE_HOST", "https://cloud.langfuse.com")

print("환경 변수 로드 결과:")
print(f"  ANTHROPIC_API_KEY: {'설정됨 ✓' if ANTHROPIC_API_KEY else '없음 ✗ (.env 파일 확인 필요)'}")
print(f"  LANGFUSE_PUBLIC_KEY: {'설정됨 ✓' if LANGFUSE_PUBLIC_KEY else '없음 (Langfuse 섹션 스킵 가능)'}")
print(f"  LANGFUSE_SECRET_KEY: {'설정됨 ✓' if LANGFUSE_SECRET_KEY else '없음 (Langfuse 섹션 스킵 가능)'}")

환경 변수 로드 결과:
  ANTHROPIC_API_KEY: 설정됨 ✓
  LANGFUSE_PUBLIC_KEY: 없음 (Langfuse 섹션 스킵 가능)
  LANGFUSE_SECRET_KEY: 없음 (Langfuse 섹션 스킵 가능)


In [ ]:
# 핵심 라이브러리 임포트
from langchain.schema import Document
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'AppleGothic'  # macOS 한글 폰트
# Windows/Linux: matplotlib.rcParams['font.family'] = 'NanumGothic'
matplotlib.rcParams['axes.unicode_minus'] = False

print("라이브러리 로드 완료 ✓")

---
## 섹션 3. 샘플 문서 준비

한국어 AI/개발 기술 문서 20개를 직접 정의합니다.
각 문서는 `content`(본문)와 `metadata`(카테고리, ID)를 가집니다.

또한 Recall@10 측정을 위한 **테스트 질의와 정답 문서 ID**를 함께 정의합니다.

In [ ]:
# 한국어 기술 문서 20개 정의
raw_docs = [
    # LLM/AI 관련
    {"id": "doc_01", "content": "GPT-4o는 OpenAI의 최신 멀티모달 모델로, 텍스트, 이미지, 오디오를 동시에 처리할 수 있습니다. API 가격은 input $5/1M tokens, output $15/1M tokens입니다.", "category": "llm"},
    {"id": "doc_02", "content": "Claude 3.5 Sonnet은 Anthropic이 개발한 대형 언어 모델입니다. 코딩, 분석, 창작 작업에서 뛰어난 성능을 보이며, 200K 토큰의 컨텍스트 창을 지원합니다.", "category": "llm"},
    {"id": "doc_03", "content": "Llama 3은 Meta가 공개한 오픈소스 LLM입니다. 8B, 70B, 405B 파라미터 버전이 있으며, 상업적 사용이 가능한 커뮤니티 라이선스를 제공합니다.", "category": "llm"},
    {"id": "doc_04", "content": "임베딩 모델은 텍스트를 고차원 벡터로 변환하는 모델입니다. text-embedding-3-small은 OpenAI의 경량 임베딩 모델로 1536 차원 벡터를 생성합니다.", "category": "embedding"},
    {"id": "doc_05", "content": "한국어 임베딩을 위해 jhgan/ko-sroberta-multitask 모델을 많이 사용합니다. 768 차원 벡터를 생성하며 KorSTS 벤치마크에서 우수한 성능을 보입니다.", "category": "embedding"},
    # RAG 관련
    {"id": "doc_06", "content": "RAG(Retrieval-Augmented Generation)는 외부 문서를 검색하여 LLM의 답변 품질을 향상시키는 기법입니다. 환각(Hallucination)을 줄이고 최신 정보를 반영할 수 있습니다.", "category": "rag"},
    {"id": "doc_07", "content": "청킹(Chunking)은 긴 문서를 작은 조각으로 나누는 과정입니다. 청크 크기는 보통 256~512 토큰으로 설정하며, 오버랩은 20~50 토큰을 권장합니다.", "category": "rag"},
    {"id": "doc_08", "content": "리랭킹(Reranking)은 초기 검색 결과를 Cross-Encoder 모델로 재정렬하는 기법입니다. BM25나 벡터 검색의 상위 결과를 더 정교하게 필터링할 수 있습니다.", "category": "rag"},
    {"id": "doc_09", "content": "하이브리드 검색은 BM25 키워드 검색과 벡터 검색을 결합하는 방식입니다. RRF(Reciprocal Rank Fusion) 알고리즘으로 두 검색 결과를 통합하면 Recall이 크게 향상됩니다.", "category": "rag"},
    {"id": "doc_10", "content": "Self-RAG는 LLM이 검색 필요성을 스스로 판단하고, 검색 결과의 관련성과 답변 지지 여부를 평가하는 고급 RAG 기법입니다.", "category": "rag"},
    # 벡터DB 관련
    {"id": "doc_11", "content": "Chroma는 오픈소스 벡터 데이터베이스로, Python에서 바로 사용할 수 있는 임베디드 모드를 지원합니다. 로컬 개발 및 프로토타이핑에 적합합니다.", "category": "vectordb"},
    {"id": "doc_12", "content": "Elasticsearch 8.x는 kNN 벡터 검색과 BM25 텍스트 검색을 네이티브로 지원합니다. rank.rrf 기능으로 두 검색을 쉽게 결합할 수 있습니다.", "category": "vectordb"},
    {"id": "doc_13", "content": "Pinecone은 완전 관리형 벡터 데이터베이스 서비스입니다. 서버리스 플랜에서 무료로 시작할 수 있으며, 수백만 벡터를 밀리초 단위로 검색합니다.", "category": "vectordb"},
    {"id": "doc_14", "content": "FAISS(Facebook AI Similarity Search)는 Meta가 개발한 고성능 벡터 유사도 검색 라이브러리입니다. HNSW, IVF 등 다양한 인덱스 알고리즘을 지원합니다.", "category": "vectordb"},
    # LangChain 관련
    {"id": "doc_15", "content": "LangChain은 LLM 애플리케이션 개발을 위한 Python/JS 프레임워크입니다. Chain, Agent, Memory, Retriever 등의 추상화를 제공하여 복잡한 AI 파이프라인을 쉽게 구축할 수 있습니다.", "category": "langchain"},
    {"id": "doc_16", "content": "LangChain EnsembleRetriever는 여러 Retriever를 가중치와 함께 결합합니다. 내부적으로 RRF 알고리즘을 사용하여 최종 순위를 계산합니다.", "category": "langchain"},
    {"id": "doc_17", "content": "LangSmith는 LangChain의 LLM 애플리케이션 디버깅 및 모니터링 플랫폼입니다. 트레이스, 평가, 데이터셋 관리 기능을 제공합니다.", "category": "langchain"},
    # 관측성/평가
    {"id": "doc_18", "content": "Langfuse는 LLM 애플리케이션을 위한 오픈소스 관측성 플랫폼입니다. 트레이스, 스팬, 스코어 기능으로 RAG 파이프라인의 품질을 모니터링할 수 있습니다.", "category": "observability"},
    {"id": "doc_19", "content": "Recall@K는 검색 품질 평가 지표입니다. 전체 관련 문서 중 상위 K개 결과 안에 포함된 관련 문서의 비율을 나타냅니다. RAG에서는 Recall@10을 주로 사용합니다.", "category": "evaluation"},
    {"id": "doc_20", "content": "RAGAS는 RAG 파이프라인 평가 프레임워크입니다. Faithfulness, Answer Relevancy, Context Precision, Context Recall 네 가지 지표로 RAG 품질을 자동 평가합니다.", "category": "evaluation"},
]

# LangChain Document 객체로 변환
docs = [
    Document(
        page_content=d["content"],
        metadata={"id": d["id"], "category": d["category"]}
    )
    for d in raw_docs
]

print(f"문서 수: {len(docs)}개")
print(f"\n예시 문서:")
print(f"  ID: {docs[8].metadata['id']}")
print(f"  카테고리: {docs[8].metadata['category']}")
print(f"  내용: {docs[8].page_content[:80]}...")

In [ ]:
# 테스트 질의 및 정답 문서 정의
# 각 질의에 대해 관련 문서 ID를 지정합니다
test_queries = [
    {
        "query": "GPT-4o API 가격은 얼마인가요?",
        "relevant_ids": ["doc_01"],
        "type": "keyword"  # 정확한 키워드 매칭이 필요
    },
    {
        "query": "한국어 텍스트를 벡터로 변환하는 모델 추천",
        "relevant_ids": ["doc_05", "doc_04"],
        "type": "semantic"  # 의미적 유사성 검색
    },
    {
        "query": "환각 문제를 줄이는 방법",
        "relevant_ids": ["doc_06"],
        "type": "semantic"
    },
    {
        "query": "BM25와 벡터 검색을 함께 사용하는 방법",
        "relevant_ids": ["doc_09", "doc_16"],
        "type": "mixed"
    },
    {
        "query": "Elasticsearch kNN 하이브리드 검색",
        "relevant_ids": ["doc_12", "doc_09"],
        "type": "keyword"
    },
    {
        "query": "문서를 작은 조각으로 나누는 전략",
        "relevant_ids": ["doc_07"],
        "type": "semantic"
    },
    {
        "query": "LLM 파이프라인 모니터링 도구",
        "relevant_ids": ["doc_18", "doc_17"],
        "type": "semantic"
    },
    {
        "query": "RAG 검색 품질 측정 지표 Recall",
        "relevant_ids": ["doc_19", "doc_20"],
        "type": "mixed"
    },
    {
        "query": "오픈소스 무료 벡터 데이터베이스",
        "relevant_ids": ["doc_11", "doc_14"],
        "type": "semantic"
    },
    {
        "query": "Anthropic Claude 컨텍스트 길이",
        "relevant_ids": ["doc_02"],
        "type": "keyword"
    },
]

print(f"테스트 질의 수: {len(test_queries)}개")
print(f"  - keyword 유형: {sum(1 for q in test_queries if q['type'] == 'keyword')}개")
print(f"  - semantic 유형: {sum(1 for q in test_queries if q['type'] == 'semantic')}개")
print(f"  - mixed 유형: {sum(1 for q in test_queries if q['type'] == 'mixed')}개")

---
## 섹션 4. BM25 검색 구현

**BM25(Best Match 25)**는 단어 빈도(TF)와 역문서빈도(IDF)를 기반으로 한 전통적인 키워드 검색 알고리즘입니다.

LangChain의 `BM25Retriever`는 내부적으로 `rank-bm25` 라이브러리를 사용합니다.

In [ ]:
from langchain_community.retrievers import BM25Retriever

# BM25 Retriever 생성
bm25_retriever = BM25Retriever.from_documents(docs)
bm25_retriever.k = 10  # 상위 10개 반환

print("BM25 Retriever 생성 완료 ✓")

# 테스트: keyword 유형 쿼리
test_query = "GPT-4o API 가격은 얼마인가요?"
bm25_results = bm25_retriever.invoke(test_query)

print(f"\n쿼리: '{test_query}'")
print(f"BM25 검색 결과 (상위 5개):")
for i, doc in enumerate(bm25_results[:5], 1):
    print(f"  {i}. [{doc.metadata['id']}] {doc.page_content[:60]}...")

In [ ]:
# BM25의 한계: 의미적 쿼리 테스트
semantic_query = "환각 문제를 줄이는 방법"
bm25_semantic_results = bm25_retriever.invoke(semantic_query)

print(f"쿼리: '{semantic_query}'")
print(f"정답 문서: doc_06 (RAG는 환각을 줄인다고 설명)")
print(f"\nBM25 검색 결과 (상위 5개):")
for i, doc in enumerate(bm25_semantic_results[:5], 1):
    is_correct = "✓ 정답" if doc.metadata['id'] == 'doc_06' else ""
    print(f"  {i}. [{doc.metadata['id']}] {doc.page_content[:60]}... {is_correct}")

result_ids = [d.metadata['id'] for d in bm25_semantic_results[:10]]
print(f"\n정답 포함 여부 (Recall@10): {'doc_06' in result_ids}")

---
## 섹션 5. 벡터 검색 구현

**[벡터 검색 프로세스]**
```mermaid
flowchart LR
    Q([사용자 쿼리]):::query --> E1(임베딩 모델):::model
    E1 --> QV[쿼리 벡터 공간]:::vector
    D([문서 DB]):::doc --> E2(임베딩 모델):::model
    E2 --> DV[문서 벡터 공간]:::vector
    QV --> SIM{유사도 계산}:::sim
    DV --> SIM
    SIM --> R([최종 검색 결과]):::result

    classDef query fill:#4285F4,color:#fff,stroke-width:2px
    classDef doc fill:#EA4335,color:#fff,stroke-width:2px
    classDef vector fill:#FBBC05,color:#000,stroke-width:2px
    classDef model fill:#9C27B0,color:#fff,stroke-width:2px
    classDef sim fill:#FF9800,color:#fff,stroke-width:2px
    classDef result fill:#34A853,color:#fff,stroke-width:2px,font-weight:bold
```

**Dense Retrieval**은 텍스트를 임베딩 벡터로 변환하고 코사인 유사도로 검색합니다.

- 임베딩 모델: `jhgan/ko-sroberta-multitask` (한국어 특화, 768차원)
- 벡터 DB: `Chroma` (로컬 인메모리)

> **참고:** 최초 실행 시 모델 다운로드(약 300MB)가 필요합니다.

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

print("임베딩 모델 로드 중... (최초 실행 시 다운로드 필요)")

# 한국어 임베딩 모델 로드
embeddings = HuggingFaceEmbeddings(
    model_name="jhgan/ko-sroberta-multitask",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

print("임베딩 모델 로드 완료 ✓")

# ChromaDB에 문서 인덱싱
print("문서 벡터화 및 인덱싱 중...")
vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    collection_name="hybrid_search_demo"
)

# 벡터 Retriever 생성
vector_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 10}
)

print(f"인덱싱 완료: {len(docs)}개 문서 ✓")

In [ ]:
# 벡터 검색 테스트
print("=" * 60)
print("[테스트 1] keyword 유형 쿼리 (벡터 검색의 약점)")
print("=" * 60)

keyword_query = "GPT-4o API 가격은 얼마인가요?"
vec_results_kw = vector_retriever.invoke(keyword_query)

print(f"\n쿼리: '{keyword_query}'")
print(f"상위 5개 결과:")
for i, doc in enumerate(vec_results_kw[:5], 1):
    is_correct = "✓ 정답" if doc.metadata['id'] == 'doc_01' else ""
    print(f"  {i}. [{doc.metadata['id']}] {doc.page_content[:60]}... {is_correct}")

print("\n" + "=" * 60)
print("[테스트 2] semantic 유형 쿼리 (벡터 검색의 강점)")
print("=" * 60)

semantic_query = "환각 문제를 줄이는 방법"
vec_results_sm = vector_retriever.invoke(semantic_query)

print(f"\n쿼리: '{semantic_query}'")
print(f"상위 5개 결과:")
for i, doc in enumerate(vec_results_sm[:5], 1):
    is_correct = "✓ 정답" if doc.metadata['id'] == 'doc_06' else ""
    print(f"  {i}. [{doc.metadata['id']}] {doc.page_content[:60]}... {is_correct}")

---
## 섹션 6. EnsembleRetriever로 RRF 하이브리드 결합

**[하이브리드 검색 아키텍처 (EnsembleRetriever)]**
```mermaid
flowchart LR
    Q([질의]):::query --> BM25[BM25 Retrieval]:::bm25
    Q --> VEC[Vector Retrieval]:::vec
    BM25 --> R1(Ranked List 1):::dlist
    VEC --> R2(Ranked List 2):::dlist
    R1 --> RRF{RRF Fusion}:::rrf
    R2 --> RRF
    RRF --> F([최종 검색 결과]):::final

    classDef query fill:#4285F4,color:#fff,stroke-width:2px
    classDef bm25 fill:#EA4335,color:#fff,stroke-width:2px
    classDef vec fill:#FBBC05,color:#000,stroke-width:2px
    classDef dlist fill:#f5f5f5,color:#333,stroke-width:2px
    classDef rrf fill:#0f3460,color:#fff,font-weight:bold,stroke-width:2px
    classDef final fill:#34A853,color:#fff,stroke-width:2px,font-weight:bold
```

**Reciprocal Rank Fusion (RRF)** 공식:

$$\text{RRF}(d) = \sum_{r \in R} \frac{1}{k + \text{rank}_r(d)}$$

LangChain의 `EnsembleRetriever`가 이 공식을 내부적으로 구현합니다.

In [ ]:
from langchain.retrievers import EnsembleRetriever

# EnsembleRetriever: BM25 + 벡터 검색을 RRF로 결합
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.5, 0.5]  # 동일 가중치 (기본값)
)

print("EnsembleRetriever 생성 완료 ✓")
print(f"  - BM25 가중치: 0.5")
print(f"  - 벡터 가중치: 0.5")

# 하이브리드 검색 테스트
print("\n" + "=" * 60)
print("하이브리드 검색 결과")
print("=" * 60)

for test in test_queries[:3]:
    results = ensemble_retriever.invoke(test["query"])
    result_ids = [d.metadata['id'] for d in results[:10]]
    
    hit = any(rid in result_ids for rid in test['relevant_ids'])
    print(f"\n[{test['type']}] {test['query']}")
    print(f"  정답 ID: {test['relevant_ids']}")
    print(f"  검색 결과 상위 5: {result_ids[:5]}")
    print(f"  정답 포함: {'✓' if hit else '✗'}")

---
## 섹션 7. Recall@10 측정 함수 구현

**Recall@K** = (상위 K개 결과 중 관련 문서 수) / (전체 관련 문서 수)

예: 관련 문서가 2개이고, 상위 10개 결과 중 1개가 관련 문서라면 Recall@10 = 0.5

In [ ]:
def compute_recall_at_k(retriever, queries: list, k: int = 10) -> dict:
    """
    Retriever의 Recall@K를 계산합니다.
    
    Args:
        retriever: LangChain Retriever 객체
        queries: [{"query": str, "relevant_ids": list, "type": str}] 형식
        k: 상위 K개 결과 평가
    
    Returns:
        {
            "overall": float,      # 전체 Recall@K
            "by_type": dict,       # 유형별 Recall@K
            "per_query": list      # 질의별 결과
        }
    """
    per_query_results = []
    
    for q in queries:
        results = retriever.invoke(q["query"])
        result_ids = [doc.metadata['id'] for doc in results[:k]]
        
        # 관련 문서 중 상위 K에 포함된 수
        relevant_found = sum(
            1 for rid in q["relevant_ids"] if rid in result_ids
        )
        recall = relevant_found / len(q["relevant_ids"])
        
        per_query_results.append({
            "query": q["query"],
            "type": q["type"],
            "recall": recall,
            "relevant_found": relevant_found,
            "total_relevant": len(q["relevant_ids"]),
        })
    
    # 전체 평균
    overall_recall = sum(r["recall"] for r in per_query_results) / len(per_query_results)
    
    # 유형별 평균
    by_type = {}
    for q_type in ["keyword", "semantic", "mixed"]:
        type_results = [r for r in per_query_results if r["type"] == q_type]
        if type_results:
            by_type[q_type] = sum(r["recall"] for r in type_results) / len(type_results)
    
    return {
        "overall": overall_recall,
        "by_type": by_type,
        "per_query": per_query_results
    }

print("Recall@K 측정 함수 정의 완료 ✓")
print("\n함수 사용법:")
print("  result = compute_recall_at_k(retriever, test_queries, k=10)")
print("  print(result['overall'])  # 전체 Recall@10")

---
## 섹션 8. BM25 vs 벡터 vs Hybrid 성능 비교 실험

세 가지 검색 방식의 Recall@10을 측정하고 비교합니다.

In [ ]:
print("성능 비교 실험 시작...")
print("=" * 60)

# 1. BM25 단독
print("[1/3] BM25 검색 평가 중...")
bm25_results = compute_recall_at_k(bm25_retriever, test_queries, k=10)

# 2. 벡터 단독
print("[2/3] 벡터 검색 평가 중...")
vector_results = compute_recall_at_k(vector_retriever, test_queries, k=10)

# 3. Hybrid (RRF)
print("[3/3] 하이브리드 검색 평가 중...")
hybrid_results = compute_recall_at_k(ensemble_retriever, test_queries, k=10)

print("\n" + "=" * 60)
print("전체 Recall@10 결과")
print("=" * 60)
print(f"  BM25 단독:      {bm25_results['overall']:.3f}")
print(f"  벡터 단독:      {vector_results['overall']:.3f}")
print(f"  Hybrid (RRF):  {hybrid_results['overall']:.3f}")

print("\n유형별 Recall@10:")
for q_type in ["keyword", "semantic", "mixed"]:
    bm25_t = bm25_results['by_type'].get(q_type, 0)
    vec_t = vector_results['by_type'].get(q_type, 0)
    hyb_t = hybrid_results['by_type'].get(q_type, 0)
    print(f"  [{q_type:8s}] BM25={bm25_t:.2f}  벡터={vec_t:.2f}  Hybrid={hyb_t:.2f}")

In [ ]:
# 가중치 튜닝 실험
print("가중치 조합별 Recall@10:")
print("-" * 40)

weight_experiments = [
    (0.3, 0.7),
    (0.5, 0.5),
    (0.7, 0.3),
]

weight_results = []
for bm25_w, vec_w in weight_experiments:
    retriever = EnsembleRetriever(
        retrievers=[bm25_retriever, vector_retriever],
        weights=[bm25_w, vec_w]
    )
    result = compute_recall_at_k(retriever, test_queries, k=10)
    weight_results.append({
        "label": f"BM25={bm25_w} / 벡터={vec_w}",
        "recall": result["overall"]
    })
    print(f"  BM25={bm25_w}, 벡터={vec_w}  →  Recall@10 = {result['overall']:.3f}")

best = max(weight_results, key=lambda x: x['recall'])
print(f"\n최적 가중치: {best['label']} (Recall@10 = {best['recall']:.3f})")

---
## 섹션 9. Elasticsearch Hybrid Query 구현 (선택사항)

Docker로 Elasticsearch가 실행 중인 경우에만 실행하세요.

```bash
# Docker로 ES 실행 (최초 1회)
docker run -d \
  --name es-hybrid \
  -e "discovery.type=single-node" \
  -e "xpack.security.enabled=false" \
  -p 9200:9200 \
  elasticsearch:8.13.0
```

> ES가 없다면 이 섹션을 건너뛰세요. BM25+ChromaDB 조합으로도 동일한 개념을 학습할 수 있습니다.

In [ ]:
import requests

ES_URL = "http://localhost:9200"
USE_ELASTICSEARCH = False  # ES가 실행 중인 경우 True로 변경

# ES 연결 확인
try:
    response = requests.get(ES_URL, timeout=2)
    if response.status_code == 200:
        USE_ELASTICSEARCH = True
        version = response.json().get('version', {}).get('number', 'unknown')
        print(f"Elasticsearch 연결 성공 ✓ (버전: {version})")
    else:
        print(f"Elasticsearch 응답 오류: {response.status_code}")
except Exception:
    print("Elasticsearch 미실행 — ES 섹션을 건너뜁니다.")
    print("ES를 사용하려면 위의 Docker 명령어를 실행 후 USE_ELASTICSEARCH = True로 변경하세요.")

In [ ]:
# Elasticsearch Hybrid 검색 구현
# USE_ELASTICSEARCH = True인 경우에만 실행됩니다

if USE_ELASTICSEARCH:
    from elasticsearch import Elasticsearch
    from langchain_elasticsearch import ElasticsearchStore
    
    es_client = Elasticsearch(ES_URL)
    INDEX_NAME = "hybrid_search_ep01"
    
    # 인덱스 생성 (벡터 필드 포함)
    if es_client.indices.exists(index=INDEX_NAME):
        es_client.indices.delete(index=INDEX_NAME)
    
    es_client.indices.create(
        index=INDEX_NAME,
        body={
            "mappings": {
                "properties": {
                    "content": {"type": "text", "analyzer": "standard"},
                    "content_vector": {
                        "type": "dense_vector",
                        "dims": 768,
                        "index": True,
                        "similarity": "cosine"
                    },
                    "doc_id": {"type": "keyword"},
                    "category": {"type": "keyword"}
                }
            }
        }
    )
    
    # 문서 인덱싱 (임베딩 포함)
    print("Elasticsearch에 문서 인덱싱 중...")
    for i, doc in enumerate(docs):
        vector = embeddings.embed_query(doc.page_content)
        es_client.index(
            index=INDEX_NAME,
            id=doc.metadata['id'],
            document={
                "content": doc.page_content,
                "content_vector": vector,
                "doc_id": doc.metadata['id'],
                "category": doc.metadata['category']
            }
        )
    
    es_client.indices.refresh(index=INDEX_NAME)
    print(f"인덱싱 완료: {len(docs)}개 문서 ✓")
    
    # Elasticsearch Hybrid Query (BM25 + kNN + RRF)
    def es_hybrid_search(query: str, k: int = 10) -> list:
        query_vector = embeddings.embed_query(query)
        
        response = es_client.search(
            index=INDEX_NAME,
            body={
                "query": {
                    "match": {
                        "content": {"query": query, "boost": 1.0}
                    }
                },
                "knn": {
                    "field": "content_vector",
                    "query_vector": query_vector,
                    "k": k,
                    "num_candidates": 100,
                    "boost": 1.0
                },
                "rank": {
                    "rrf": {
                        "window_size": 50,
                        "rank_constant": 60
                    }
                },
                "size": k
            }
        )
        
        return [
            hit["_source"]["doc_id"]
            for hit in response["hits"]["hits"]
        ]
    
    # ES Hybrid 검색 테스트
    test_q = test_queries[0]
    es_result_ids = es_hybrid_search(test_q["query"])
    print(f"\nES Hybrid 검색 테스트")
    print(f"  쿼리: {test_q['query']}")
    print(f"  결과: {es_result_ids[:5]}")
    print(f"  정답 포함: {any(rid in es_result_ids for rid in test_q['relevant_ids'])}")

else:
    print("Elasticsearch를 사용하지 않습니다. 다음 섹션으로 이동하세요.")

---
## 섹션 10. Langfuse CallbackHandler 통합

Langfuse로 검색 파이프라인을 추적하고 품질 점수를 기록합니다.

`.env` 파일에 `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY`가 없다면 이 섹션은 건너뜁니다.

In [ ]:
USE_LANGFUSE = bool(LANGFUSE_PUBLIC_KEY and LANGFUSE_SECRET_KEY)

if USE_LANGFUSE:
    from langfuse import Langfuse
    from langfuse.callback import CallbackHandler
    
    langfuse_client = Langfuse(
        public_key=LANGFUSE_PUBLIC_KEY,
        secret_key=LANGFUSE_SECRET_KEY,
        host=LANGFUSE_HOST
    )
    
    # 연결 확인
    langfuse_client.auth_check()
    print("Langfuse 연결 성공 ✓")
    print(f"  Host: {LANGFUSE_HOST}")
else:
    print("Langfuse API 키 미설정 — 추적 없이 실행됩니다.")
    print(".env 파일에 LANGFUSE_PUBLIC_KEY, LANGFUSE_SECRET_KEY를 추가하면 추적이 활성화됩니다.")

In [ ]:
def run_experiment_with_langfuse(
    retriever,
    experiment_name: str,
    queries: list,
    k: int = 10
) -> float:
    """
    실험을 실행하고 Langfuse에 결과를 기록합니다.
    Langfuse 미설정 시 추적 없이 Recall@K만 반환합니다.
    """
    if USE_LANGFUSE:
        # Langfuse 트레이스 생성
        trace = langfuse_client.trace(
            name=experiment_name,
            metadata={
                "k": k,
                "num_queries": len(queries)
            }
        )
        handler = CallbackHandler(
            trace_id=trace.id,
            public_key=LANGFUSE_PUBLIC_KEY,
            secret_key=LANGFUSE_SECRET_KEY,
            host=LANGFUSE_HOST
        )
        config = {"callbacks": [handler]}
    else:
        trace = None
        config = {}
    
    # 검색 실행 및 Recall 계산
    per_query_results = []
    for q in queries:
        results = retriever.invoke(q["query"], config=config)
        result_ids = [doc.metadata['id'] for doc in results[:k]]
        relevant_found = sum(1 for rid in q['relevant_ids'] if rid in result_ids)
        recall = relevant_found / len(q['relevant_ids'])
        per_query_results.append(recall)
    
    overall_recall = sum(per_query_results) / len(per_query_results)
    
    # Langfuse에 점수 기록
    if trace:
        trace.score(
            name=f"recall@{k}",
            value=overall_recall,
            comment=f"experiment: {experiment_name}"
        )
        langfuse_client.flush()
        print(f"  Langfuse에 점수 기록 완료: recall@{k} = {overall_recall:.3f}")
    
    return overall_recall


# 세 가지 방식 실험 실행 (Langfuse 추적 포함)
print("Langfuse 추적과 함께 실험 실행...")
print("=" * 60)

experiments = [
    (bm25_retriever, "ep01_bm25_only"),
    (vector_retriever, "ep01_vector_only"),
    (ensemble_retriever, "ep01_hybrid_rrf_0.5_0.5"),
]

experiment_scores = {}
for retriever, name in experiments:
    print(f"\n실험: {name}")
    score = run_experiment_with_langfuse(retriever, name, test_queries, k=10)
    experiment_scores[name] = score
    print(f"  Recall@10 = {score:.3f}")

print("\n" + "=" * 60)
print("실험 완료!")
if USE_LANGFUSE:
    print(f"  Langfuse 대시보드에서 결과 확인: {LANGFUSE_HOST}")

---
## 섹션 11. 결과 시각화

세 가지 검색 방식의 Recall@10을 matplotlib으로 시각화합니다.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 전체 Recall@10 비교
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('EP01 Hybrid Search - Recall@10 비교', fontsize=14, fontweight='bold')

# --- 차트 1: 전체 Recall@10 ---
ax1 = axes[0]
methods = ['BM25\n단독', '벡터 검색\n단독', 'Hybrid\n(RRF)']
scores = [
    bm25_results['overall'],
    vector_results['overall'],
    hybrid_results['overall']
]
colors = ['#4285F4', '#EA4335', '#34A853']
bars = ax1.bar(methods, scores, color=colors, width=0.5, edgecolor='white', linewidth=1.5)

# 점수 레이블
for bar, score in zip(bars, scores):
    ax1.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.01,
        f'{score:.3f}',
        ha='center', va='bottom', fontsize=12, fontweight='bold'
    )

ax1.set_ylim(0, 1.1)
ax1.set_ylabel('Recall@10', fontsize=11)
ax1.set_title('전체 Recall@10', fontsize=12)
ax1.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5, label='최대값')
ax1.legend(fontsize=9)
ax1.grid(axis='y', alpha=0.3)
ax1.spines[['top', 'right']].set_visible(False)

# --- 차트 2: 쿼리 유형별 Recall@10 ---
ax2 = axes[1]
query_types = ['keyword', 'semantic', 'mixed']
type_labels = ['키워드형', '의미형', '혼합형']
x = np.arange(len(query_types))
width = 0.25

bm25_by_type = [bm25_results['by_type'].get(t, 0) for t in query_types]
vec_by_type = [vector_results['by_type'].get(t, 0) for t in query_types]
hyb_by_type = [hybrid_results['by_type'].get(t, 0) for t in query_types]

b1 = ax2.bar(x - width, bm25_by_type, width, label='BM25', color='#4285F4', alpha=0.85)
b2 = ax2.bar(x, vec_by_type, width, label='벡터', color='#EA4335', alpha=0.85)
b3 = ax2.bar(x + width, hyb_by_type, width, label='Hybrid (RRF)', color='#34A853', alpha=0.85)

ax2.set_xticks(x)
ax2.set_xticklabels(type_labels, fontsize=10)
ax2.set_ylim(0, 1.2)
ax2.set_ylabel('Recall@10', fontsize=11)
ax2.set_title('쿼리 유형별 Recall@10', fontsize=12)
ax2.legend(fontsize=9)
ax2.grid(axis='y', alpha=0.3)
ax2.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('recall_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("차트 저장: recall_comparison.png")

In [ ]:
# 질의별 상세 결과 테이블
print("\n질의별 상세 Recall@10:")
print("=" * 90)
print(f"{'질의':<40} {'유형':<8} {'BM25':>6} {'벡터':>6} {'Hybrid':>8}")
print("-" * 90)

for i, q in enumerate(test_queries):
    bm25_r = bm25_results['per_query'][i]['recall']
    vec_r = vector_results['per_query'][i]['recall']
    hyb_r = hybrid_results['per_query'][i]['recall']
    
    # Hybrid가 가장 좋은 경우 표시
    best = max(bm25_r, vec_r, hyb_r)
    hyb_mark = " ★" if hyb_r == best and hyb_r > 0 else ""
    
    query_short = q['query'][:38] + ".." if len(q['query']) > 38 else q['query']
    print(f"{query_short:<40} {q['type']:<8} {bm25_r:>6.2f} {vec_r:>6.2f} {hyb_r:>6.2f}{hyb_mark}")

print("=" * 90)
print(f"{'평균':<40} {'':8} {bm25_results['overall']:>6.3f} {vector_results['overall']:>6.3f} {hybrid_results['overall']:>8.3f}")
print("\n★ = 해당 질의에서 Hybrid가 최고 성능")

---
## 섹션 12. Exercise

아래 두 가지 Exercise를 완료하면 EP01을 마스터한 것입니다!

### Exercise 1: 새로운 질의 유형 추가 실험

**목표:** 한국어 기술 질의 5개를 추가하고 세 방식의 Recall@10을 비교한다.

**과제:**
1. `test_queries` 리스트에 새로운 질의 5개를 추가하라
   - `keyword` 유형 2개, `semantic` 유형 2개, `mixed` 유형 1개
   - 각 질의에 올바른 `relevant_ids`를 지정하라
2. BM25, 벡터, Hybrid 세 방식으로 새 질의에 대한 Recall@10을 계산하라
3. 어떤 유형의 질의에서 각 방식이 실패하는지 분석하라

**힌트:** 문서가 없는 질의를 만들면 Recall이 0이 됩니다.
샘플 문서 20개 중 실제로 관련 있는 문서가 있는 질의를 작성하세요.

In [ ]:
# Exercise 1 풀이 공간
# TODO: 아래에 새로운 질의 5개를 추가하세요

additional_queries = [
    # 예시 (주석 해제 후 수정하세요):
    # {
    #     "query": "여기에 질의를 입력하세요",
    #     "relevant_ids": ["doc_XX"],
    #     "type": "keyword"  # 또는 "semantic", "mixed"
    # },
]

if additional_queries:
    all_queries = test_queries + additional_queries
    
    print("새로운 질의에 대한 Recall@10:")
    for retriever, name in [
        (bm25_retriever, "BM25"),
        (vector_retriever, "벡터"),
        (ensemble_retriever, "Hybrid")
    ]:
        result = compute_recall_at_k(retriever, additional_queries, k=10)
        print(f"  {name}: {result['overall']:.3f}")
else:
    print("additional_queries 리스트에 질의를 추가하세요!")

### Exercise 2: 새 문서 추가 + 재실험

**목표:** 문서를 추가하고 가중치를 최적화하여 전체 Recall@10을 0.9 이상으로 달성한다.

**과제:**
1. `raw_docs` 리스트에 새로운 한국어 기술 문서 5개를 추가하라
   - 기존 테스트 질의에 대한 정답 문서가 포함되도록 하라
   - 예: `test_queries`에서 현재 정답을 못 찾는 질의들의 관련 문서 추가
2. 새 문서를 포함하여 BM25와 ChromaDB를 **재초기화**하라
3. EnsembleRetriever 가중치를 `[0.2, 0.8]`, `[0.4, 0.6]`, `[0.6, 0.4]`, `[0.8, 0.2]`로 실험하라
4. 최고 Recall@10을 달성하는 가중치를 찾고 결과를 시각화하라

**보너스:** Langfuse에 각 실험을 다른 `experiment_name`으로 기록하고 대시보드에서 비교하라

In [ ]:
# Exercise 2 풀이 공간
# TODO: 새 문서 추가 및 가중치 최적화 실험

# 1단계: 새 문서 정의
new_docs = [
    # 여기에 새 문서를 추가하세요
    # {"id": "doc_21", "content": "...", "category": "..."},
]

if new_docs:
    # 2단계: 전체 문서 합치기
    all_raw_docs = raw_docs + new_docs
    all_docs = [
        Document(
            page_content=d["content"],
            metadata={"id": d["id"], "category": d["category"]}
        )
        for d in all_raw_docs
    ]
    
    # 3단계: BM25 재초기화
    new_bm25 = BM25Retriever.from_documents(all_docs)
    new_bm25.k = 10
    
    # 4단계: ChromaDB 재초기화
    new_vectorstore = Chroma.from_documents(
        documents=all_docs,
        embedding=embeddings,
        collection_name="hybrid_search_ex2"
    )
    new_vector_retriever = new_vectorstore.as_retriever(search_kwargs={"k": 10})
    
    # 5단계: 가중치 최적화
    weights_to_try = [(0.2, 0.8), (0.4, 0.6), (0.5, 0.5), (0.6, 0.4), (0.8, 0.2)]
    
    best_score = 0
    best_weights = None
    
    print("가중치 최적화 실험:")
    for bw, vw in weights_to_try:
        retriever = EnsembleRetriever(
            retrievers=[new_bm25, new_vector_retriever],
            weights=[bw, vw]
        )
        result = compute_recall_at_k(retriever, test_queries, k=10)
        score = result['overall']
        print(f"  BM25={bw:.1f}, 벡터={vw:.1f}  →  Recall@10 = {score:.3f}")
        if score > best_score:
            best_score = score
            best_weights = (bw, vw)
    
    print(f"\n최적 가중치: BM25={best_weights[0]}, 벡터={best_weights[1]}")
    print(f"최고 Recall@10: {best_score:.3f}")
    print(f"목표 달성 (≥ 0.9): {'✓' if best_score >= 0.9 else f'✗ (추가 실험 필요)'}")
else:
    print("new_docs 리스트에 새 문서를 추가하세요!")

---
## 마무리

### 이 노트북에서 배운 것

| 개념 | 핵심 포인트 |
|------|------------|
| BM25 | 키워드 기반, 빠르고 정확하지만 의미를 모름 |
| Dense Retrieval | 의미 기반, 동의어 처리하지만 키워드에 약함 |
| RRF | 절대 점수가 아닌 순위 기반 결합 → 스케일 무관 |
| EnsembleRetriever | LangChain의 RRF 구현체, weights로 가중치 조정 |
| Recall@K | 검색 품질의 핵심 지표, K=10이 RAG 실전 표준 |
| Langfuse | 실험별 점수 기록 → A/B 테스트 및 모니터링 |

### 다음 에피소드

**EP02. Reranking** — Cross-Encoder로 하이브리드 검색 결과를 더 정밀하게 재정렬하는 방법

> 참고 자료:
> - [LangChain EnsembleRetriever 문서](https://python.langchain.com/docs/modules/data_connection/retrievers/ensemble)
> - [Elasticsearch RRF 문서](https://www.elastic.co/guide/en/elasticsearch/reference/current/rrf.html)
> - [Langfuse Python SDK 문서](https://langfuse.com/docs/sdk/python)